# FracAtlas fracture classifier

In [ ]:
from pathlib import Path
import json
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")
import math
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# This works whether Jupyter starts in ai-service/ or ai-service/notebooks/.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
SERVICE_DIR = next(path for path in candidates if (path / 'app').is_dir() and (path / 'requirements.txt').exists())
PROJECT_DIR = SERVICE_DIR.parent
sys.path.insert(0, str(SERVICE_DIR))

from app.config import ARTIFACT_DIR, IMAGE_SIZE, MODEL_VERSION, SEED
from app.data import load_manifest
from app.labels import CLASS_NAMES
from app.model import build_model

DATASET_CSV = PROJECT_DIR / 'Dataset' / 'FracAtlas' / 'dataset.csv'
IMAGE_DIR = PROJECT_DIR / 'Dataset' / 'FracAtlas' / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
print('Service:', SERVICE_DIR)
print('Dataset:', DATASET_CSV)
print('TensorFlow:', tf.__version__)
print('Python:', sys.executable)
GPUS = tf.config.list_physical_devices('GPU')
print('GPUs:', GPUS)
if not GPUS:
    raise RuntimeError('GPU kernel not selected. Choose Python (FractureCare AI GPU) before training.')

## 1. Load and label the dataset

FracAtlas provides `fractured` and `fracture_count`. The application contract groups those fields into three classes.

In [ ]:
frame = load_manifest()

print(f'Usable images: {len(frame):,}')
display(frame[['image_id', 'label', 'path']].head())
display(frame['label'].value_counts().reindex(CLASS_NAMES).rename('count').to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, class_name in zip(axes, CLASS_NAMES):
    subset = frame[frame['label'] == class_name]
    if subset.empty:
        axis.text(0.5, 0.5, 'No images found', ha='center', va='center')
        axis.set_title(class_name)
        axis.axis('off')
        continue
    sample = subset.sample(1, random_state=SEED).iloc[0]
    image = cv2.imread(sample['path'], cv2.IMREAD_GRAYSCALE)
    axis.imshow(image, cmap='gray')
    axis.set_title(class_name)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 2. Create reproducible train, validation and test splits

In [ ]:
train, holdout = train_test_split(
    frame, test_size=0.2, random_state=SEED, stratify=frame['label_index']
)
validation, test = train_test_split(
    holdout, test_size=0.5, random_state=SEED, stratify=holdout['label_index']
)

train = train.reset_index(drop=True)
validation = validation.reset_index(drop=True)
test = test.reset_index(drop=True)
print(f'Train: {len(train):,} | validation: {len(validation):,} | test: {len(test):,}')
display(pd.DataFrame({
    'train': train['label'].value_counts().reindex(CLASS_NAMES),
    'validation': validation['label'].value_counts().reindex(CLASS_NAMES),
    'test': test['label'].value_counts().reindex(CLASS_NAMES),
}))

## 3. Build the TensorFlow input pipeline

Images are decoded as grayscale X-rays, converted to three channels for the CNN, resized to 224×224 and kept in the 0–255 range. The model performs its own rescaling.

In [ ]:
BATCH_SIZE = 32

def make_dataset(dataframe, shuffle=False):
    paths = dataframe['path'].to_numpy()
    labels = dataframe['label_index'].to_numpy(dtype=np.int32)

    def load(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, IMAGE_SIZE)
        return image, label

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.apply(tf.data.experimental.ignore_errors())
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train, shuffle=True).repeat()
validation_dataset = make_dataset(validation).repeat()
test_dataset = make_dataset(test)
TRAIN_STEPS = math.ceil(len(train) / BATCH_SIZE)
VALIDATION_STEPS = math.ceil(len(validation) / BATCH_SIZE)
class_weights_array = compute_class_weight(
    class_weight='balanced', classes=np.arange(len(CLASS_NAMES)), y=train['label_index'].to_numpy()
)
class_weights = {index: float(weight) for index, weight in enumerate(class_weights_array)}
print('Class weights:', class_weights)

## 4. Train the baseline model

You can change the architecture, augmentation, optimizer, epochs or class weighting in this notebook as you develop the model.

In [ ]:
model = build_model()
model.summary()

In [ ]:
EPOCHS = 20
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        ARTIFACT_DIR / 'fracture_classifier.keras', monitor='val_accuracy', save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
]
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    shuffle=False,
    steps_per_epoch=TRAIN_STEPS,
    validation_steps=VALIDATION_STEPS,
)

In [ ]:
history_frame = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_frame[['accuracy', 'val_accuracy']].plot(ax=axes[0], title='Accuracy')
history_frame[['loss', 'val_loss']].plot(ax=axes[1], title='Loss')
plt.tight_layout()
plt.show()

## 5. Evaluate on the held-out test set

In [ ]:
best_model = tf.keras.models.load_model(ARTIFACT_DIR / 'fracture_classifier.keras')
test_loss, test_accuracy = best_model.evaluate(test_dataset, verbose=0)
probabilities = best_model.predict(test_dataset, verbose=0)
predicted = probabilities.argmax(axis=1)
actual = np.concatenate([labels.numpy() for _, labels in test_dataset], axis=0)

print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
print(classification_report(actual, predicted, target_names=CLASS_NAMES, zero_division=0))

matrix = confusion_matrix(actual, predicted, labels=np.arange(len(CLASS_NAMES)))
plt.figure(figsize=(7, 5))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted class')
plt.ylabel('Actual class')
plt.title('Held-out test confusion matrix')
plt.show()

## 6. Save the service metadata

The FastAPI service reads the `.keras` model and this metadata file. Keep them together whenever you change the model.

In [ ]:
test.to_csv(ARTIFACT_DIR / 'test_manifest.csv', index=False)
metadata = {
    'modelVersion': MODEL_VERSION,
    'classes': list(CLASS_NAMES),
    'imageSize': list(IMAGE_SIZE),
    'datasetCsv': str(DATASET_CSV),
    'trainCount': int(len(train)),
    'validationCount': int(len(validation)),
    'testCount': int(len(test)),
    'classWeights': class_weights,
    'testLoss': float(test_loss),
    'testAccuracy': float(test_accuracy),
    'epochsCompleted': len(history.history['loss']),
}
(ARTIFACT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved artifacts to:', ARTIFACT_DIR)
print(json.dumps(metadata, indent=2))

## 7. Try one image

This uses the same preprocessing as the API endpoint.

In [ ]:
def predict_path(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f'Could not read image: {path}')
    image = cv2.resize(image, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
    image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    values = best_model.predict(np.expand_dims(image, axis=0), verbose=0)[0]
    return {name: float(values[index]) for index, name in enumerate(CLASS_NAMES)}

sample_path = Path(test.iloc[0]['path'])
print(sample_path.name)
print(predict_path(sample_path))